# **Welcome to the Notebook**

### Task 1 - Set up the project

Imporint the modules

In [0]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import pandas as pd
import numpy as np

from pyspark.sql.functions import concat_ws
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType

from pyspark.ml.feature import VectorAssembler, PCA
from pyspark.ml.clustering import KMeans
import plotly.express as px

In [0]:
PRODUCT_DATASET_LOCATION = "dbfs:/Volumes/workspace/default/datasets/recommender_system/products_dataset.csv"

In [0]:
raw_df = (spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(PRODUCT_DATASET_LOCATION)
)
raw_df.show()

List of 8 products recently viewed by the user.

In [0]:
recently_viewed_products = [
    'P316',
    'P333',
    'P1115',
    'P1691',
    'P1082',
    'P397',
    'P1441',
    'P1054',
]

### Task 2 - Prepare the dataset

Combine `title` and `description` Columns

In [0]:
raw_df = raw_df.withColumn("combined_text", concat_ws(" ", raw_df.title, raw_df.description))
raw_df.show()

get the combined_text column and convert it into a list

In [0]:
from sentence_transformers import SentenceTransformer
list_combined_text = (
    raw_df.select("combined_text")
          .toPandas()["combined_text"]
          .tolist()
)


model = SentenceTransformer("all-MiniLM-L6-v2")  # fast + good quality
embeddings = model.encode(list_combined_text, batch_size=32, show_progress_bar=True)

In [0]:
import pandas as pd
import numpy as np

# 1) Get original data in Pandas (keep order stable)
pdf = raw_df.toPandas().reset_index(drop=True)

# 2) Build embedding DataFrame with column names
emb_array = np.array(embeddings)  # shape (n_rows, dim)
features_column_names = [f"embedding_{i}" for i in range(emb_array.shape[1])]

emb_df = pd.DataFrame(emb_array, columns=features_column_names)

# 3) Combine with original
pdf_combined = pd.concat([pdf, emb_df], axis=1)

# 4) Back to Spark
df_with_embeddings = spark.createDataFrame(pdf_combined)


In [0]:
print(df_with_embeddings.columns[:5])      # original columns
print(df_with_embeddings.columns[-5:])     # embedding_XXX columns


In [0]:
# 1) Check row counts match
assert len(pdf) == emb_array.shape[0], "Row count mismatch between df and embeddings"

# 2) Spot check a few rows (text + first 3 embedding values)
sample_idx = [0, 5, 10]  # pick a few indices you like
for i in sample_idx:
    print("Row:", i)
    print("Text:", pdf.loc[i, "combined_text"][:80], "...")
    print("Embedding[:3]:", emb_array[i][:3])
    print("----")

In [0]:
import numpy as np
from pyspark.sql.types import ArrayType, FloatType

# embeddings: numpy array shape (n_rows, dim) or list of lists
emb_list = embeddings.tolist() if isinstance(embeddings, np.ndarray) else embeddings

pdf = raw_df.toPandas().reset_index(drop=True)
pdf["embedding"] = emb_list

data_df = spark.createDataFrame(
    pdf,
    schema=raw_df.schema.add("embedding", ArrayType(FloatType()))
)

In [0]:
data_df.show()

### Task 3 - Cluster products using K-means

Apply K-Means Clustering with 5 Clusters on the `features` Column

In [0]:
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans

# 1) Convert array -> Vector (required by Spark ML)
to_vector = F.udf(lambda x: Vectors.dense(x), VectorUDT())

df_vec = data_df.withColumn("features", to_vector(F.col("embedding")))

# 2) Fit KMeans
kmeans = KMeans(k=5, seed=42, featuresCol="features", predictionCol="cluster")
model = kmeans.fit(df_vec)

# 3) Assign clusters
clustered = model.transform(df_vec)

# 4) See cluster distribution
display(clustered.groupBy("cluster").count().orderBy("cluster"))

In [0]:
clustered.show()


### Task 4 - Visualize the clusters

Let's reduce the dimensionality of our features for visualization purpose

`512 dimensions => 2 dimensions`

In [0]:
from pyspark.ml.feature import PCA
from pyspark.sql import functions as F

# Fit PCA with k=2
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(clustered)

# Transform to get 2D coords
pca_results = pca_model.transform(clustered)



In [0]:
pca_df = pca_results.select("product_id",  "pca_features", "cluster").toPandas()
pca_df["x"] = pca_df["pca_features"].apply(lambda x: x[0])
pca_df["y"] = pca_df["pca_features"].apply(lambda x: x[1])
pca_df.head()


Let's plot the Clusters

In [0]:
def plot_clusters(pca_df, num_clusters=5):
    """
    Plots a 2D visualization of clusters using Plotly Express.

    Parameters:
    - pca_df (DataFrame): A Pandas DataFrame containing columns 'x', 'y', and 'cluster'.
      'x' and 'y' are the 2D PCA components, and 'cluster' indicates the cluster label.
    - num_clusters (int): The number of unique clusters to display.
    - recently_viewed_df (DataFrame, optional): DataFrame with 'x' and 'y' coordinates for recently viewed products.

    This function creates an interactive scatter plot where each point is colored according to its cluster.
    Recently viewed products are marked as black crosses if provided.

    Returns:
    - fig (Figure): The Plotly figure object for the plot.
    """

    # Create the base cluster plot
    fig = px.scatter(
        pca_df,
        x='x',
        y='y',
        opacity=0.6,
        size_max=4,
        color= pca_df.cluster.astype(str),
        title='2D Visualization of Clusters with Recently Viewed Products',
        labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
        category_orders={'cluster': list(range(num_clusters))},
        # show the product id in the tooltip
        hover_data={'product_id': True}

    )

    # Update layout to add legend title and adjust plot settings
    fig.update_layout(legend_title_text='Clusters', legend=dict(x=1, y=1), width=600, height=500)

    return fig

fig = plot_clusters(pca_df)
fig.show()

### Task 5 - Highlight recently viewed products

In [0]:
print("The user has recently viewed the following products: ", recently_viewed_products)

Let's have a look at the records in our `clustered_data` dataframe related to the recently viewed products.

### Task 6 - Recommend products based on recently viewed products

Let's have a look at the recently viewed products titles

In [0]:
print(recently_viewed_products)

Let's see the distinct clusters of the recenetly viewed products.

In [0]:
recent_df = clustered.where(F.col("product_id").isin(recently_viewed_products))
recent_df.show()
unique_cluster =  recent_df.select("cluster").distinct().collect()
print(unique_cluster)


Let's find the possible products for the recommendation.

In [0]:
recent_df.select("title").collect()

Let's perform a groupby and generate a list of product IDs that can be recommended for each of the clusters.

In [0]:
clustered.filter(clustered['cluster'].isin(unique_cluster)).show()
# get the product ids from the cluster

In [0]:
# write a python function to display the recommendations
def display_recommendations(row):
  # find the title of the product in df
  product_ids = row['random_recommendations']
  cluster = row.cluster

  titles = data. \
          filter(data["product_id"]. \
          isin(product_ids)).select("title").collect()

  print("\n")
  print("Recommendations for Cluster:", cluster)
  for title in titles:
    print(title[0])

recommendations_df.apply(display_recommendations, axis=1)